# 02 – Klassifisering av økttype med maskinlæring

**Problem:** Turer med Runna-tittel har kjent type (rolig/lang/tempo/hard). Turer uten
tittel – og alle turene til folk som ikke bruker Runna – må gjettes.

**Idé:** Bruk turene med tittel som treningsdata (fasit), og lær en modell som
predikerer typen bare fra målingene (distanse, puls, tempo, runder).

**Mål:** Slå de håndskrevne reglene, som treffer på 65 %.

## 1. Data

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

from lopedashboard.analyse import FEATURES, FEATURES_UTEN_RUNDER, TYPER, klassifiser, lag_features, les_garmin

warnings.filterwarnings("ignore")

df = les_garmin("../data/Activities_løping.csv")
# bruk_modell=False: vi vil se reglene, ikke en modell vi allerede har trent
df = klassifiser(df, bruk_modell=False)
X_alle = lag_features(df)

# Treningsdata: turer der typen står i tittelen (fasit), som ikke er småturer, og har alle målinger
merket = (df["kilde"] == "tittel") & df["gyldig"] & X_alle.notna().all(axis=1)
X = X_alle[merket]
y = df.loc[merket, "okttype"]

print(f"{len(y)} turer med fasit")
y.value_counts()

## 2. Features

Alle features er **relative** til løperprofilen (`lag_profil()` i `analyse.py`). Da kan modellen
brukes på andre løpere, som har annen makspuls, annet tempo og annet volum enn deg.

| feature | betydning |
|---|---|
| `km_rel` | turlengde / din typiske turlengde |
| `andel_av_uke` | turlengde / ditt ukesnitt (siste 12 aktive uker) |
| `puls_rel` | snittpuls / makspuls (oppgitt av løperen, ellers høyeste registrerte) |
| `makspuls_rel` | turens makspuls / makspuls |
| `puls_spenn` | (makspuls − snittpuls) / høyeste puls – stor når pulsen svinger (intervaller) |
| `tempo_rel` | bakkejustert tempo / ditt typiske tempo (under 1 = raskere enn vanlig) |
| `runder_per_km` | mange korte runder = intervaller |

In [ ]:
X.assign(okttype=y).groupby("okttype").mean().round(2)

In [ ]:
FARGER = {"rolig": "#2a9d8f", "lang": "#457b9d", "tempo": "#f4a261", "hard": "#e63946"}

fig, akser = plt.subplots(1, 2, figsize=(13, 5))
for t in TYPER:
    d = X[y == t]
    akser[0].scatter(d["puls_rel"], d["runder_per_km"], color=FARGER[t], label=t, s=60)
    akser[1].scatter(d["km_rel"], d["tempo_rel"], color=FARGER[t], label=t, s=60)
akser[0].set(xlabel="puls_rel", ylabel="runder_per_km", title="Puls og runder")
akser[1].set(xlabel="km_rel", ylabel="tempo_rel (lavere = raskere)", title="Lengde og tempo")
akser[0].legend()
plt.show()

## 3. Utgangspunkt: de håndskrevne reglene

Vi later som titlene ikke finnes og ser hvor ofte reglene gjetter riktig.

In [ ]:
uten_tittel = klassifiser(df.assign(tittel=""), bruk_modell=False)
regel_pred = uten_tittel.loc[merket, "okttype"]
print(f"Regler: {accuracy_score(y, regel_pred):.0%} riktig")
ConfusionMatrixDisplay.from_predictions(y, regel_pred, labels=TYPER)
plt.title("Håndskrevne regler")
plt.show()

## 4. Sammenlign modeller med leave-one-out kryssvalidering

Med bare ~40 turer blir et vanlig trenings-/testsplitt for tilfeldig.
**Leave-one-out:** tren på alle turer unntatt én, prediker den ene, og gjenta for hver tur.
Da blir hver tur testet nøyaktig én gang, av en modell som ikke har sett den.

In [ ]:
modeller = {
    "Beslutningstre (dybde 3)": DecisionTreeClassifier(max_depth=3, random_state=0),
    "Beslutningstre (dybde 5)": DecisionTreeClassifier(max_depth=5, random_state=0),
    "Logistisk regresjon": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "k-nærmeste naboer (k=3)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=0),
}

resultater = {"Håndskrevne regler": accuracy_score(y, regel_pred)}
prediksjoner = {}
for navn, modell in modeller.items():
    pred = cross_val_predict(modell, X, y, cv=LeaveOneOut())
    prediksjoner[navn] = pred
    resultater[navn] = accuracy_score(y, pred)

resultater = pd.Series(resultater).sort_values()
ax = resultater.plot.barh(figsize=(8, 4), color=["#999" if n == "Håndskrevne regler" else "#457b9d" for n in resultater.index])
ax.set_xlim(0, 1)
ax.set_xlabel("andel riktig (leave-one-out)")
for i, v in enumerate(resultater):
    ax.text(v + 0.01, i, f"{v:.0%}", va="center")
plt.show()

## 5. Den beste modellen: et lite beslutningstre

Et tre med dybde 3 er både blant de mest treffsikre og det enkleste å forstå.
Et dypere tre gjør det *dårligere* her – det lærer seg detaljer i de få treningsturene
som ikke gjelder generelt (overtilpasning).

In [ ]:
BESTE = "Beslutningstre (dybde 3)"
ConfusionMatrixDisplay.from_predictions(y, prediksjoner[BESTE], labels=TYPER)
plt.title(BESTE)
plt.show()

In [ ]:
tre = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(tre, feature_names=FEATURES, class_names=tre.classes_, filled=True, rounded=True, fontsize=10, ax=ax)
plt.show()

In [ ]:
# Hvilke features bruker treet mest?
pd.Series(tre.feature_importances_, index=FEATURES).sort_values().plot.barh(figsize=(7, 3))
plt.title("Feature importance")
plt.show()

## 6. Et problem: lærer modellen løping, eller lærer den Runna?

Treet bruker `runder_per_km` som aller viktigste regel. Men Runna-økter lager en ny runde
for hvert steg i økta (oppvarming, drag, pause …), mens vanlige turer bare får en
automatisk runde per km. Sjekk forskjellen:

In [ ]:
uten_tittel_maske = (df["kilde"] != "tittel") & df["gyldig"]
print(f"Runder per km, turer MED Runna-tittel:  {X_alle.loc[merket, 'runder_per_km'].median():.2f}")
print(f"Runder per km, turer UTEN Runna-tittel: {X_alle.loc[uten_tittel_maske, 'runder_per_km'].median():.2f}")

For en løper uten Runna (og som ikke trykker på rundeknappen) blir `runder_per_km` ≈ 1 på
*alle* turer – og da ville treet kalt nesten alt for «rolig» eller «lang».
Hvor gode er modellene hvis vi fjerner rundene?

In [ ]:
X_uten = X[FEATURES_UTEN_RUNDER]
uten_runder = {}
for navn, modell in modeller.items():
    pred = cross_val_predict(modell, X_uten, y, cv=LeaveOneOut())
    uten_runder[navn] = accuracy_score(y, pred)

pd.DataFrame({
    "med runder": resultater.drop("Håndskrevne regler"),
    "uten runder": pd.Series(uten_runder),
}).sort_values("med runder").plot.barh(figsize=(8, 4), xlim=(0, 1))
plt.xlabel("andel riktig (leave-one-out)")
plt.show()

**Konklusjon:** Rundene er det sterkeste signalet – men bare når de følger økta.
Uten dem klarer modellene rundt 60–70 % (logistisk regresjon er best, og `andel_av_uke` hjelper).

Løsningen i `analyse.py`:
- **To modeller:** et beslutningstre *med* runder og logistisk regresjon *uten* runder.
- `runder_er_informative()` sjekker om minst 10 % av personens turer har ≥ 1,3 runder per km.
  Hvis ja → modellen med runder. Hvis nei → modellen uten.

Dette er et klassisk eksempel på at en modell kan se god ut på testdata, men lære
noe annet enn det du tror (her: «er dette en Runna-økt?»). Derfor er det lurt å
se *hva* modellen bruker, ikke bare hvor ofte den har rett.

## 7. Hva sier modellen om turene uten tittel?

Sammenligning av reglene og modellen for turene som ikke har Runna-tittel.
Der de er uenige, kan du sjekke selv hva som var riktig.

In [ ]:
uten = (df["kilde"] != "tittel") & df["gyldig"] & X_alle.notna().all(axis=1)
sammenlign = df.loc[uten, ["dato", "tittel", "km", "snittpuls", "runder"]].copy()
sammenlign["regel"] = df.loc[uten, "okttype"]
sammenlign["modell"] = tre.predict(X_alle[uten])
sammenlign["uenige"] = sammenlign["regel"] != sammenlign["modell"]
sammenlign

## 8. Ta modellene i bruk

Begge modellene trenes og lagres med `scripts/tren_modell.py`. Da bruker `klassifiser()`
i `analyse.py` dem automatisk for alle turer uten tittel (kilde = `"modell"`), både i ditt
dashboard og for andre som laster opp sine data.

```bash
uv run scripts/tren_modell.py
```

**Begrensninger å være ærlig om:**
- Bare ~40 turer, og bare 7 tempoturer – resultatet kan endre seg mye med mer data.
- Trent på én løper. Bør testes på noen andres data før man stoler på den.
- For løpere uten strukturerte runder er treffsikkerheten bare ~70 %.
- Fasiten er Runna sin plan, ikke hvordan turen faktisk ble løpt.

## Ideer videre

- [ ] Tren på nytt når Strava-dataene fra før 2025 kommer (flere turer = bedre modell)
- [ ] Teste modellen på en venns data
- [ ] Features fra pulsen sekund for sekund (FIT-filer): hvor mye pulsen svinger i løpet av turen.
      Det skiller intervaller fra jevne turer *uten* å være avhengig av runder.